# Setup and Dependencies

In [ ]:
!pip install pandas numpy matplotlib scikit-learn xgboost lightgbm scipy seaborn pyarrow openpyxl

# Data Processing, Model Training and Evaluation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from google.colab import files
import io
import os
import re

from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.tree import DecisionTreeRegressor
import xgboost as xgb
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ================= 0. DIRECTORY & STYLE SETUP =================
# Create directory structure for exports (resultv2)
os.makedirs('resultv2/figures', exist_ok=True)
os.makedirs('resultv2/tables', exist_ok=True)
os.makedirs('resultv2/csv', exist_ok=True)

# Strict style configuration for scientific publication (IEEE Standard)
plt.style.use('default')
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'serif'],
    'axes.linewidth': 1.0,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'axes.grid': False,
    'figure.dpi': 100,      # Display DPI
    'savefig.dpi': 600,     # Export DPI
    'pdf.fonttype': 42,
    'ps.fonttype': 42
})

def save_figure(fig, filename):
    """Saves the figure in both high-res PNG and PDF formats."""
    safe_name = re.sub(r'[^A-Za-z0-9_]', '_', filename)
    fig.savefig(f'resultv2/figures/{safe_name}.pdf', format='pdf', bbox_inches='tight')
    fig.savefig(f'resultv2/figures/{safe_name}.png', format='png', dpi=600, bbox_inches='tight')

# ================= 1. HELPER FUNCTIONS =================

def analyze_dataset(df, stage_name):
    """Generates a comprehensive dataset analysis report with unhashable safety."""
    print(f"\n--- Dataset Analysis: {stage_name} ---")

    # Safe duplicate check (handles unhashable types like ndarray/lists)
    try:
        duplicates = df.duplicated().sum()
    except TypeError:
        duplicates = df.astype(str).duplicated().sum()

    analysis = {
        'Number of Rows': len(df),
        'Number of Columns': len(df.columns),
        'Missing Values': df.isna().sum().sum(),
        'Duplicated Rows': duplicates,
        'Memory Usage (MB)': df.memory_usage(deep=True).sum() / (1024 ** 2)
    }

    col_stats = []
    for col in df.columns:
        # Safe unique count (handles unhashable types)
        try:
            unique_count = df[col].nunique()
        except TypeError:
            unique_count = df[col].astype(str).nunique()

        col_stats.append({
            'Column': col,
            'Data Type': str(df[col].dtype),
            'Unique Values': unique_count,
            'Missing Percentage': (df[col].isna().sum() / len(df)) * 100,
            'Mean': df[col].mean() if pd.api.types.is_numeric_dtype(df[col]) else np.nan,
            'Std': df[col].std() if pd.api.types.is_numeric_dtype(df[col]) else np.nan,
            'Min': df[col].min() if pd.api.types.is_numeric_dtype(df[col]) else np.nan,
            'Max': df[col].max() if pd.api.types.is_numeric_dtype(df[col]) else np.nan
        })

    df_col_stats = pd.DataFrame(col_stats)

    # Save to CSV
    df_col_stats.to_csv(f'resultv2/csv/dataset_analysis_{stage_name.lower().replace(" ", "_")}.csv', index=False)

    print(f"Rows: {analysis['Number of Rows']} | Columns: {analysis['Number of Columns']}")
    print(f"Missing Values: {analysis['Missing Values']} | Duplicates: {analysis['Duplicated Rows']}")
    display(df_col_stats)

    return analysis

def plot_distributions(df, stage_name):
    """Plots histograms and boxplots for numeric variables safely (No Titles for IEEE)."""
    numeric_cols = df.select_dtypes(include=np.number).columns
    for col in numeric_cols:
        fig, axes = plt.subplots(1, 2, figsize=(10, 4))

        # Drop NAs and ensure 1D numeric data
        clean_data = df[col].dropna()
        if len(clean_data) > 0 and pd.api.types.is_numeric_dtype(clean_data):
            # Histogram
            sns.histplot(clean_data, bins=40, color='#808080', edgecolor='black', ax=axes[0])
            axes[0].set_ylabel('Frequency', fontsize=10)
            axes[0].set_xlabel(f'{col}', fontsize=10)
            axes[0].spines['top'].set_visible(False)
            axes[0].spines['right'].set_visible(False)

            # Boxplot
            sns.boxplot(x=clean_data, color='#D3D3D3', flierprops={'marker': 'o', 'markersize': 3}, ax=axes[1])
            axes[1].set_xlabel(f'{col}', fontsize=10)
            axes[1].spines['top'].set_visible(False)
            axes[1].spines['right'].set_visible(False)

            plt.tight_layout()
            save_figure(fig, f'distribution_{col}_{stage_name.lower().replace(" ", "_")}')
        plt.close()

# ================= 2. DATA LOAD =================
print("=== Strict Temporal Validation (Train: 2018-2023 | Test: 2024-2025) ===")

# NOME DO ARQUIVO (já presente no ambiente do Colab)
file_name = '1997_2025_Doencas_Autoimunes.parquet'

print(f"Lendo o arquivo diretamente da memória: {file_name}...")

if file_name.endswith('.parquet'):
    df_raw = pd.read_parquet(file_name)
else:
    df_raw = pd.read_excel(file_name)

print(f"\n[SUCCESS] Data loaded! Original rows: {len(df_raw)}")

# Raw Data Analysis
raw_analysis = analyze_dataset(df_raw, "Raw Dataset")
plot_distributions(df_raw, "Raw")

# ================= 3. PRE-PROCESSING & AGGREGATION =================
print("\n[Processing] Cleaning, filtering (Modern Era >= 2018), and Monthly Aggregation...")

if 'ano_compra' in df_raw.columns:
    col_year, col_month = 'ano_compra', 'mes_compra'
else:
    col_year, col_month = 'ano', 'mes'

# Filter from 2018 onwards (Removing 2015-2017 gap)
df_processed = df_raw[df_raw[col_year] >= 2018].copy()

# Ensure pure text in textual columns to avoid grouping errors
for col in ['uf', 'codigo_br', 'cid10']:
    if col in df_processed.columns:
        df_processed[col] = df_processed[col].astype(str)

# Monthly Aggregation
df_monthly = df_processed.groupby([col_year, col_month, 'uf', 'codigo_br', 'cid10'], as_index=False).agg(
    quantidade_total=('quantidade', 'sum'),
    gasto_total_mensal=('gasto_total', 'sum'),
    preco_medio_mensal=('preco_unitario', 'mean'),
    licitacoes_no_mes=('quantidade', 'count')
)

df_monthly = df_monthly.dropna(subset=['gasto_total_mensal'])

# Factorization of categorical variables (keeping unique mappings for regional analysis)
factorize_mappings = {}
for col in ['uf', 'codigo_br', 'cid10']:
    factorized, uniques = pd.factorize(df_monthly[col])
    df_monthly[col] = factorized
    factorize_mappings[col] = uniques

# Processed Data Analysis
processed_analysis = analyze_dataset(df_monthly, "Processed Dataset")
plot_distributions(df_monthly, "Processed")

# Comparative Table: Raw vs Processed
df_comparison = pd.DataFrame([raw_analysis, processed_analysis], index=['Raw Dataset', 'Processed Dataset'])
df_comparison.to_csv('resultv2/csv/dataset_comparison.csv')
print("\n=== Dataset Transformation Comparison ===")
display(df_comparison)

# Features and Target
features = [col_year, col_month, 'quantidade_total', 'preco_medio_mensal', 'licitacoes_no_mes', 'uf', 'codigo_br', 'cid10']
X = df_monthly[features]
y = df_monthly['gasto_total_mensal']

# ================= 4. OUT-OF-TIME SPLIT =================
# Train: Past (<= 2023) | Test: Real Future (>= 2024)
train_idx = df_monthly[col_year] <= 2023
test_idx = df_monthly[col_year] >= 2024

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print(f"\n[SUCCESS] Train Instances (2018-2023): {len(X_train)}")
print(f"[SUCCESS] Test Instances (2024-2025): {len(X_test)}")

# ================= 5. TRAINING AND EVALUATION =================
models = {
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'Extra Trees': ExtraTreesRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'XGBoost': xgb.XGBRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'LightGBM': lgb.LGBMRegressor(n_estimators=100, random_state=42, n_jobs=-1, verbose=-1)
}

metrics = {}
regional_metrics_list = []
epsilon = 1e-8

print("\n[Training] Executing models with strict temporal validation...")

for name, model in models.items():
    print(f" -> Processing {name}...")
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    # Calculate Metrics
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))

    y_true_safe = np.where(y_test == 0, epsilon, y_test)
    preds_safe = np.where(preds == 0, epsilon, preds)

    mape = np.mean(np.abs((y_test - preds) / y_true_safe)) * 100
    smape = 100 * np.mean(2 * np.abs(preds - y_test) / (np.abs(y_test) + np.abs(preds_safe)))

    metrics[name] = {'MAE': mae, 'RMSE': rmse, 'MAPE': mape, 'SMAPE': smape}
    safe_name = name.replace(" ", "_").lower()

    # --- 5.1 Residual Analysis (No Titles) ---
    residuals = y_test - preds
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # Distribution
    axes[0].hist(residuals, bins=40, color='white', edgecolor='black', hatch='\\\\')
    axes[0].set_xlabel('Residuals (Actual - Predicted)', fontsize=11)
    axes[0].set_ylabel('Frequency', fontsize=11)

    # Residuals vs Predicted
    axes[1].scatter(preds, residuals, alpha=0.4, color='black', s=10)
    axes[1].axhline(0, color='red', linestyle='--', linewidth=1.2)
    axes[1].set_xlabel('Predicted Values', fontsize=11)
    axes[1].set_ylabel('Residuals', fontsize=11)

    # QQ Plot
    stats.probplot(residuals, dist="norm", plot=axes[2])
    axes[2].get_lines()[0].set_color('black')
    axes[2].get_lines()[1].set_color('red')
    # Probplot forces a title internally, let's remove it explicitly
    axes[2].set_title('')
    axes[2].set_xlabel('Theoretical Quantiles', fontsize=11)
    axes[2].set_ylabel('Ordered Values', fontsize=11)

    for ax in axes:
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    plt.tight_layout()
    save_figure(fig, f'residuals_{safe_name}')
    plt.close()

    # --- 5.2 Actual vs Predicted (No Titles) ---
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(y_test, preds, alpha=0.4, color='black', s=10)
    min_val = min(np.min(y_test), np.min(preds))
    max_val = max(np.max(y_test), np.max(preds))

    # Identity line and discrete grid
    ax.plot([min_val, max_val], [min_val, max_val], color='black', linestyle='--', linewidth=1.5, label='Identity Line')
    ax.set_aspect('equal', 'box')
    ax.grid(color='gray', linestyle=':', linewidth=0.5, alpha=0.5)

    ax.set_xlabel('Actual Expenditure - 2024/2025 (BRL)', fontsize=11)
    ax.set_ylabel('Predicted Expenditure (BRL)', fontsize=11)
    ax.legend(frameon=False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    save_figure(fig, f'actual_vs_predicted_{safe_name}')
    plt.close()

    # --- 5.3 Temporal Trend Analysis (No Titles) ---
    df_temp = X_test.copy()
    df_temp['Actual'] = y_test
    df_temp['Predicted'] = preds

    time_trend = df_temp.groupby([col_year, col_month]).sum().reset_index()
    time_trend = time_trend.sort_values([col_year, col_month])
    time_trend['Date'] = time_trend[col_year].astype(str) + '-' + time_trend[col_month].astype(str).str.zfill(2)

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(time_trend['Date'], time_trend['Actual'], color='black', label='Actual (2024-2025)', marker='o', markersize=4)
    ax.plot(time_trend['Date'], time_trend['Predicted'], color='#606060', linestyle='--', label=f'Predicted ({name})', marker='s', markersize=4)

    ax.set_ylabel('Monthly Budget (BRL)', fontsize=11)
    ax.set_xlabel('Test Horizon (Year-Month)', fontsize=11)

    skip = max(1, len(time_trend) // 10)
    ax.set_xticks(time_trend['Date'][::skip])
    plt.xticks(rotation=45)
    ax.legend(loc='upper left', frameon=False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(color='gray', linestyle=':', linewidth=0.5, alpha=0.3, axis='y')

    plt.tight_layout()
    save_figure(fig, f'temporal_trend_{safe_name}')
    plt.close()

    # --- 5.4 Feature Importance (No Titles) ---
    if hasattr(model, 'feature_importances_'):
        importances = model.feature_importances_
        df_imp = pd.DataFrame({'Feature': features, 'Importance': importances}).sort_values(by='Importance', ascending=True)

        # Save CSV
        df_imp.to_csv(f'resultv2/csv/feature_importance_{safe_name}.csv', index=False)

        fig, ax = plt.subplots(figsize=(6, 4))
        ax.barh(df_imp['Feature'], df_imp['Importance'], color='black', edgecolor='black', alpha=0.7)
        ax.set_xlabel('Importance Score', fontsize=11)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

        plt.tight_layout()
        save_figure(fig, f'feature_importance_{safe_name}')
        plt.close()

        print(f"\n[{name}] Feature Importance computed and saved.")
        display(df_imp.sort_values(by='Importance', ascending=False).head(3))

    # --- 5.5 Regional Analysis Evaluation ---
    if 'uf' in df_temp.columns:
        # Reconstruct state names from factorized mapping safely
        df_temp['Region'] = df_temp['uf'].map(lambda x: factorize_mappings['uf'][x] if x < len(factorize_mappings['uf']) else "Unknown")
        regional_group = df_temp.groupby('Region').apply(
            lambda g: pd.Series({
                'MAE': mean_absolute_error(g['Actual'], g['Predicted']),
                'RMSE': np.sqrt(mean_squared_error(g['Actual'], g['Predicted'])),
                'MAPE': np.mean(np.abs((g['Actual'] - g['Predicted']) / np.where(g['Actual']==0, epsilon, g['Actual']))) * 100,
                'SMAPE': 100 * np.mean(2 * np.abs(g['Predicted'] - g['Actual']) / (np.abs(g['Actual']) + np.where(g['Predicted']==0, epsilon, np.abs(g['Predicted']))))
            })
        ).reset_index()
        regional_group['Model'] = name
        regional_metrics_list.append(regional_group)

# ================= 6. GLOBAL COMPARISON =================
print("\n[Processing] Generating global benchmarking charts...")
df_metrics = pd.DataFrame(metrics).T
df_metrics.index.name = 'Model'

# Export overall metrics to CSV
df_metrics.to_csv('resultv2/tables/overall_metrics_comparison.csv')

# --- 6.1 Individual Bar Charts (MAE, RMSE, MAPE, SMAPE) (No Titles) ---
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()
bar_colors = ['#333333', '#666666', '#999999', '#CCCCCC', '#E5E5E5']

for i, metric in enumerate(['MAE', 'RMSE', 'MAPE', 'SMAPE']):
    ax = axes[i]
    values = df_metrics[metric]
    bars = ax.bar(values.index, values, color=bar_colors, edgecolor='black', alpha=0.9)

    for bar in bars:
        yval = bar.get_height()
        label = f'{yval:.1f}%' if 'MAPE' in metric else f'{yval:.2e}'
        ax.annotate(label, xy=(bar.get_x() + bar.get_width() / 2, yval),
                    xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=9)

    ax.set_ylabel(metric, fontsize=11)
    ax.set_xticklabels(values.index, rotation=25, ha='right')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
save_figure(fig, 'global_metrics_barcharts')
plt.close()

# --- 6.2 Metrics Heatmap (No Titles) ---
fig, ax = plt.subplots(figsize=(8, 5))
# Normalize columns for heatmap visualization
df_metrics_norm = (df_metrics - df_metrics.min()) / (df_metrics.max() - df_metrics.min())
sns.heatmap(df_metrics_norm, annot=df_metrics, fmt=".2f", cmap="Greys", cbar=True, ax=ax, linewidths=.5, linecolor='black')
plt.tight_layout()
save_figure(fig, 'global_metrics_heatmap')
plt.close()

# --- 6.3 Overall Ranking ---
# Lower values mean better performance
overall_ranking = df_metrics.rank(ascending=True).mean(axis=1).sort_values()
df_ranking = pd.DataFrame({'Average Rank Score': overall_ranking})
df_ranking.to_csv('resultv2/tables/overall_ranking.csv')

# ================= 7. REGIONAL VISUALIZATIONS =================
if regional_metrics_list:
    print("\n[Processing] Generating regional evaluation charts...")
    df_regional_all = pd.concat(regional_metrics_list, ignore_index=True)
    df_regional_all.to_csv('resultv2/csv/regional_metrics_full.csv', index=False)

    # Pivot SMAPE for Heatmap
    regional_smape_pivot = df_regional_all.pivot(index='Region', columns='Model', values='SMAPE')

    # Regional Heatmap (No Titles)
    fig, ax = plt.subplots(figsize=(10, max(4, len(regional_smape_pivot) * 0.3)))
    sns.heatmap(regional_smape_pivot, annot=True, fmt=".1f", cmap="Greys", linewidths=.5, linecolor='black', ax=ax)
    plt.tight_layout()
    save_figure(fig, 'regional_smape_heatmap')
    plt.close()

    # Regional Bar Chart (Average SMAPE across regions for the best model) (No Titles)
    best_model_name = df_ranking.index[0]
    best_regional = df_regional_all[df_regional_all['Model'] == best_model_name].sort_values('SMAPE')

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(best_regional['Region'], best_regional['SMAPE'], color='gray', edgecolor='black')
    ax.set_ylabel('SMAPE (%)', fontsize=11)
    plt.xticks(rotation=45, ha='right')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    save_figure(fig, 'regional_best_model_barchart')
    plt.close()
else:
    print("\n[INFO] Regional identifier column not found post-processing. Skipping regional charts.")

# ================= 8. FINAL OUTPUTS =================
print("\n=== Final Comparative Summary (Out-of-Time Validation) ===")
display(df_metrics.round(3))

print("\n=== Overall Model Ranking (Lower Rank = Better) ===")
display(df_ranking)

print("\n[COMPLETED] Pipeline V2 Execution Successful! All titles removed from plots. Saved in 'results/' folder.")

# Export Results and Download Files

In [ ]:
import shutil
from google.colab import files
import os

# ================= 9. DOWNLOAD ALL RESULTS V2 =================
print("=== Zipping and Downloading Result V2 ===")

folder_to_zip = 'results'
output_filename = 'ieee_experiment_resultv2.zip'

if os.path.exists(folder_to_zip):
    print(f"[Processing] Compressing the '{folder_to_zip}' directory...")

    # Creates the zip file
    shutil.make_archive(output_filename.replace('.zip', ''), 'zip', folder_to_zip)

    print(f"[SUCCESS] Archive created! Initiating download for {output_filename}...")
    files.download(output_filename)
else:
    print(f"[ERROR] The folder '{folder_to_zip}' does not exist. Please run the main pipeline first.")